## Rule 1 and Rule 2: Extending Join Patterns

Based on patterns from the BIRD dev set, we extend each pattern by adding one table. Rule 1 focuses on queries with N+1 tables, where N is the largest number of tables in any pattern, while Rule 2 handles queries with 2 to N tables. To extend the patterns, we add the table most connected to the existing nodes.

For each extended pattern, we generate its equivalent patterns. If no equivalent is isomorphic to the original pattern, the least connected one is added to the output; otherwise, we skip the pattern.

The output is then sorted by the number of tables (nodes), followed by join relations (edges), cyclicity, and centrality scores. Patterns are divided into two groups: Rule 1 for patterns with N+1 nodes, and Rule 2 for patterns with 2 to N nodes.

### Important Note

**_To generate the queries, you must include one development database ID in the first cell at a time. Run all the cells except for the last two each time._**


## 1. Retrieving the dev patterns for each database
The following cells retrieve the join patterns extracted from the development set queries of the BIRD benchmark. These patterns are stored in the `dev_join_patterns` folder, which can be accessed here: [BIRD Join Patterns](https://drive.google.com/drive/folders/1Cip2-kP5JKPFtDry40Y-SorNh0Rcps57).

The folder contains data for each development database, including:
1. **Images** of the join patterns.
2. **Pickle files** storing the patterns (subgraphs) grouped by the number of nodes along with their details (nodes, edges, etc)


In [131]:
import os
import pickle
from pathlib import Path

def retrieve_all_dev_patterns(folder_path):
    """
    Retrieve all subgraphs from the pickle files in the folder.
    
    Args:
        folder_path (str): The path to the folder containing the .pkl files.
        
    Returns:
        list: A list of all subgraphs from the folder.
    """
    folder = Path(folder_path)
    all_subgraphs = []

    for file_name in folder.glob('join_patterns_*.pkl'):
        try:
            with file_name.open('rb') as f:
                join_patterns = pickle.load(f)
                subgraphs = join_patterns.get('subgraphs', [])
                all_subgraphs.extend(subgraphs)
        except (ValueError, KeyError, pickle.UnpicklingError) as e:
            print(f"Skipping file {file_name.name}: {e}")

    return all_subgraphs

# Example usage:
db_id = 'toxicology'
folder_path = f'dev_join_patterns/{db_id}/'
# Retrieve all subgraphs from the folder
database_subgraphs = retrieve_all_dev_patterns(folder_path)
print(f"Found {len(database_subgraphs)} subgraphs.")


Found 13 subgraphs.


In this cell, the schema graph for each database is retrieved from the corresponding pickle file stored in the `main_graphs_pickles` folder under `dev_dbs_unnested`. The folder can be accessed here: [Schema Graphs](https://drive.google.com/drive/folders/18a1XT3zZFJO-k28OQT7Cqcm9VlZ7qljS).


In [132]:
import os
import networkx as nx
import matplotlib.pyplot as plt
import pickle
from datetime import datetime

# Load the schema (main graph) from the pickle file
schema_path = f'dev_dbs_unnested/main_graphs_pickles/{db_id}_graph.pkl'
with open(schema_path, 'rb') as f:
    schema = pickle.load(f)



def display_and_save_subgraph(schema, subgraph, subgraph_number, rule, time_period, db_id):
    """
    Display and save a subgraph within the main schema.

    Args:
        schema (nx.Graph): The main graph schema.
        subgraph (nx.Graph): The subgraph to display.
        subgraph_number (int): The number of the subgraph for labeling.
        rule (str): The rule this subgraph belongs to (either 'rule_1' or 'rule_2').
        time_period (str): Either 'pre_rule' or 'post_rule' indicating the time period.
        db_id (str): The database ID.
    """
    # Create the directory structure: images/rule_x/db_id/pre_rule or post_rule
    dir_path = os.path.join('images', rule, db_id, time_period)
    os.makedirs(dir_path, exist_ok=True)  # Ensure the directory exists even if the subgraph is None or empty

    # Create a figure for the plot
    plt.figure(figsize=(10, 10))
    
    # Draw the entire schema in light gray
    pos = nx.spring_layout(schema)  # Node positions using spring layout
    nx.draw(schema, pos, with_labels=True, node_color='lightgray', edge_color='black', 
            node_size=1000, font_size=10)

    # Check if the subgraph is None or has nodes
    if subgraph is not None and subgraph.number_of_nodes() > 0:  # Draw the subgraph only if it contains nodes
        # Draw the subgraph nodes and edges
        nx.draw_networkx_nodes(subgraph, pos, node_color='lightblue', node_size=1000)
        nx.draw_networkx_edges(subgraph, pos, edge_color='blue', width=2)

        # Create edge labels for the subgraph, fallback to schema if needed
        edge_labels = {
            (u, v): edge_data.get('label', schema[u][v].get('label', 'No Label'))
            for u, v, edge_data in subgraph.edges(data=True)
        }

        # Draw the edge labels
        nx.draw_networkx_edge_labels(subgraph, pos, edge_labels=edge_labels, font_color='red', font_size=7)

    # Add title indicating the subgraph number
    plt.title(f'Subgraph {subgraph_number}')

    # Generate a unique filename using timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    image_filename = f'subgraph_{subgraph_number}_{timestamp}.png'
    image_path = os.path.join(dir_path, image_filename)

    # Save the image
    plt.savefig(image_path)
    plt.close()  # Close the plot to free up memory

    print(f'Subgraph {subgraph_number} saved at: {image_path}')


In [133]:
def save_pre_rule_subgraphs(schema, subgraphs, rule_name, db_id):
    """
    Save and display pre-rule subgraphs for a specific rule.

    Parameters:
    - schema: The main schema graph.
    - subgraphs: List of subgraphs to process.
    - rule_name: Name of the rule (e.g., 'rule_1', 'rule_2').
    - db_id: Database identifier for saving images.
    """
    subgraphs_sorted = sorted(subgraphs, key=lambda x: x.number_of_nodes())  # Sort by number of nodes
    for i, subgraph in enumerate(subgraphs_sorted, start=1):
        display_and_save_subgraph(
            schema=schema,
            subgraph=subgraph,
            subgraph_number=i,
            rule=rule_name,
            time_period='pre_rule',
            db_id=db_id
        )

# Save pre-rule subgraphs for rule_1 and rule_2
save_pre_rule_subgraphs(schema=schema, subgraphs=database_subgraphs, rule_name='rule_1', db_id=db_id)
save_pre_rule_subgraphs(schema=schema, subgraphs=database_subgraphs, rule_name='rule_2', db_id=db_id)


Subgraph 1 saved at: images/rule_1/toxicology/pre_rule/subgraph_1_20241211_173111.png
Subgraph 2 saved at: images/rule_1/toxicology/pre_rule/subgraph_2_20241211_173111.png
Subgraph 3 saved at: images/rule_1/toxicology/pre_rule/subgraph_3_20241211_173111.png
Subgraph 4 saved at: images/rule_1/toxicology/pre_rule/subgraph_4_20241211_173111.png
Subgraph 5 saved at: images/rule_1/toxicology/pre_rule/subgraph_5_20241211_173111.png
Subgraph 6 saved at: images/rule_1/toxicology/pre_rule/subgraph_6_20241211_173111.png
Subgraph 7 saved at: images/rule_1/toxicology/pre_rule/subgraph_7_20241211_173111.png
Subgraph 8 saved at: images/rule_1/toxicology/pre_rule/subgraph_8_20241211_173111.png
Subgraph 9 saved at: images/rule_1/toxicology/pre_rule/subgraph_9_20241211_173111.png
Subgraph 10 saved at: images/rule_1/toxicology/pre_rule/subgraph_10_20241211_173111.png
Subgraph 11 saved at: images/rule_1/toxicology/pre_rule/subgraph_11_20241211_173112.png
Subgraph 12 saved at: images/rule_1/toxicology/pre

## 2. Implementing the algorithm for Rule 1 and 2

In [134]:
import networkx as nx
from itertools import combinations

def extend_and_filter_subgraphs(pre_rule_subgraphs, schema):
    """
    Function to process subgraphs, extend them, filter for uniqueness,
    and categorize them based on specific rules.

    Parameters:
    - pre_rule_subgraphs: List of initial subgraphs in the dev set.
    - schema: The schema graph representing the full set of tables and relationships.

    Returns:
    - rule_1_patterns: Patterns with N+1 tables (maximum in dev set + 1).
    - rule_2_patterns: Patterns with 2 to N tables.
    """
    def find_candidate_table(schema, subgraph):
        """
        Find candidate tables for extending the subgraph.
        """
        subgraph_nodes = set(subgraph.nodes())
        candidate_tables = {}

        for node in schema.nodes():
            if node not in subgraph_nodes:
                connections_to_subgraph = list(filter(lambda n: n in subgraph_nodes, schema.neighbors(node)))
                if connections_to_subgraph:
                    candidate_tables[node] = connections_to_subgraph

        if not candidate_tables:
            return []

        def sort_key(item):
            table, connections = item
            num_connections = len(connections)
            centrality = nx.degree_centrality(schema).get(table, 0)
            return (-num_connections, -centrality)

        return sorted(candidate_tables.items(), key=sort_key)

    def is_pattern_in_list(pattern, pattern_list):
        """
        Check if a pattern or its isomorphic equivalent exists in the given list.
        """
        for existing_pattern in pattern_list:
            if nx.is_isomorphic(pattern, existing_pattern):
                return True
        return False

    def generate_connected_subgraphs(schema, nodes):
        """
        Generate all connected subgraphs from a given set of nodes.
        """
        edges = list(combinations(nodes, 2))
        valid_edges = [edge for edge in edges if schema.has_edge(*edge)]
        subgraphs = []

        for i in range(1, len(valid_edges) + 1):
            for edge_subset in combinations(valid_edges, i):
                subgraph = nx.Graph()
                subgraph.add_nodes_from(nodes)
                subgraph.add_edges_from(edge_subset)
                if nx.is_connected(subgraph):
                    subgraphs.append(subgraph)

        return subgraphs

    unique_patterns = []  # Store all unique patterns across iterations

    # Step 1: Extend each subgraph and filter for unique patterns
    for subgraph in pre_rule_subgraphs:
        candidate_tables = find_candidate_table(schema, subgraph)

        for candidate_table, connections in candidate_tables:
            # Create the extended subgraph
            extended_subgraph = subgraph.copy()
            extended_subgraph.add_node(candidate_table)
            extended_subgraph.add_edges_from([(candidate_table, conn) for conn in connections])

            # Check isomorphic conditions
            if not is_pattern_in_list(extended_subgraph, pre_rule_subgraphs):
                equivalent_patterns = generate_connected_subgraphs(schema, list(extended_subgraph.nodes()))
                valid_candidates = [
                    pattern for pattern in [extended_subgraph] + equivalent_patterns
                    if not is_pattern_in_list(pattern, pre_rule_subgraphs)
                ]

                # If valid candidates exist, choose the one with the fewest edges
                if valid_candidates:
                    chosen_pattern = min(valid_candidates, key=lambda p: len(p.edges()))
                    if not is_pattern_in_list(chosen_pattern, unique_patterns):
                        unique_patterns.append(chosen_pattern)

    # Step 2: Divide the unique patterns into Rule 1 and Rule 2 categories
    rule_1_patterns = []
    rule_2_patterns = []
    max_nodes = max(len(subgraph.nodes()) for subgraph in pre_rule_subgraphs)

    for pattern in unique_patterns:
        if len(pattern.nodes()) == max_nodes + 1:
            rule_1_patterns.append(pattern)
        elif 2 <= len(pattern.nodes()) <= max_nodes:
            rule_2_patterns.append(pattern)

    # Step 3: Sort both lists by number of edges
    rule_1_patterns.sort(key=lambda p: len(p.nodes()))
    rule_2_patterns.sort(key=lambda p: len(p.nodes()))

    return rule_1_patterns, rule_2_patterns

pre_rule_sorted_patterns = sorted(database_subgraphs, key=lambda x: x.number_of_nodes())
rule_1_patterns, rule_2_patterns = extend_and_filter_subgraphs(pre_rule_sorted_patterns, schema)

In [135]:
print(f"Rule 1 Patterns: {len(rule_1_patterns)}")
print(f"Rule 2 Patterns: {len(rule_2_patterns)}")

Rule 1 Patterns: 2
Rule 2 Patterns: 1


## 3. Partial ordering of the resulting patterns from Rule 1 and 2

In [136]:
import networkx as nx

# Compute centrality for each node in the main graph (schema)
centrality = nx.degree_centrality(schema)

def calculate_subgraph_centrality(subgraph, centrality_dict):
    """
    Calculate the average centrality for a given subgraph.
    """
    subgraph_nodes = list(subgraph.nodes())
    return sum(centrality_dict.get(node, 0) for node in subgraph_nodes) / len(subgraph_nodes)

def is_cyclic(subgraph):
    """
    Check if a given subgraph contains a cycle.
    """
    try:
        nx.find_cycle(subgraph)
        return True
    except nx.NetworkXNoCycle:
        return False

def process_rule_subgraphs(rule_patterns, rule_name, schema, db_id):
    """
    Process and display subgraphs for a given rule.
    
    Parameters:
    - rule_patterns: List of subgraphs for the rule.
    - rule_name: Name of the rule (e.g., 'rule_1', 'rule_2').
    - schema: The main schema graph.
    - db_id: Database identifier for saving images.
    """
    # Extract and calculate subgraph metrics
    rule_metrics = []
    for subgraph in rule_patterns:
        num_nodes = subgraph.number_of_nodes()  # Number of nodes (tables)
        num_connections = subgraph.number_of_edges()  # Number of connections (edges)
        cyclic = is_cyclic(subgraph)  # Check if subgraph is cyclic
        centrality_score = calculate_subgraph_centrality(subgraph, centrality)  # Calculate average centrality
        rule_metrics.append((subgraph, num_nodes, num_connections, cyclic, centrality_score))

    # Sort by number of nodes, connections, cyclic status, and centrality
    rule_metrics.sort(key=lambda x: (x[1], x[2], x[3], x[4]), reverse=True)

    # Check if metrics are empty
    if not rule_metrics:
        print(f"No new metrics available for {rule_name}.")
        # Optionally, create folders and save an image of just the schema
        display_and_save_subgraph(
            schema=schema,
            subgraph=None,  # No subgraph to display
            subgraph_number=0,
            rule=rule_name,
            time_period='post_rule',
            db_id=db_id
        )
        return

    # Display sorted metrics and save subgraphs
    for i, (subgraph, num_nodes, num_connections, cyclic, centrality_score) in enumerate(rule_metrics, start=1):
        cyclic_status = "Cyclic" if cyclic else "Acyclic"
        print(f"{rule_name.capitalize()} Subgraph {i}: {num_nodes} nodes, {num_connections} connections, {cyclic_status}, centrality: {centrality_score:.4f}")

        display_and_save_subgraph(
            schema=schema,
            subgraph=subgraph,
            subgraph_number=i,
            rule=rule_name,
            time_period='post_rule',
            db_id=db_id
        )

    # Return sorted subgraphs for further use
    return [subgraph for subgraph, *_ in rule_metrics]

# Process both rule_1 and rule_2
rule_1_sorted_patterns = process_rule_subgraphs(rule_1_patterns, 'rule_1', schema, db_id)
rule_2_sorted_patterns = process_rule_subgraphs(rule_2_patterns, 'rule_2', schema, db_id)


Rule_1 Subgraph 1: 4 nodes, 3 connections, Acyclic, centrality: 0.8333
Subgraph 1 saved at: images/rule_1/toxicology/post_rule/subgraph_1_20241211_173113.png
Rule_1 Subgraph 2: 4 nodes, 3 connections, Acyclic, centrality: 0.8333
Subgraph 2 saved at: images/rule_1/toxicology/post_rule/subgraph_2_20241211_173113.png
Rule_2 Subgraph 1: 3 nodes, 3 connections, Cyclic, centrality: 0.8889
Subgraph 1 saved at: images/rule_2/toxicology/post_rule/subgraph_1_20241211_173113.png


## 4. Generating the SQL queries with placeholders for Rule 1 and 2
In this cell, data are retrieved from a CSV file named `query_statistics.csv`, which contains statistics about the number of projection and filtering columns used in the development set for each database. The file is located in the `Bird` folder and can be accessed here: [Query Statistics](https://drive.google.com/drive/folders/1knzQu_44H92yJC9HljHHkvqS_49N9IB8).


In [137]:
import pandas as pd
import sqlglot
from sqlglot import expressions as exp
import random

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('query_statistics.csv')

def get_table_data(schema, node_name):
    """
    Retrieve table data from the schema given a node name.
    
    Parameters:
    - schema: The graph schema containing table information.
    - node_name: The name of the node (table) to retrieve data for.
    
    Returns:
    - Tuple containing table name, list of column names, and list of column types.
    """
    if node_name in schema.nodes:
        data = schema.nodes[node_name]
        return data['table_name'], data['column_names'], data['column_types']
    return None, [], []

def retrieve_projection_filter_columns(db_id, df, tables_involved):
    """
    Retrieve the projection and filtering columns along with their corresponding tables based on occurrences for a given db_id.
    
    Parameters:
    - db_id: The database ID to filter (e.g., 'card_games').
    - df: The pandas DataFrame containing the data.
    - tables_involved: List of table names involved in the current query model.
    
    Returns:
    - projection_columns_with_tables: List of tuples (table_name, projection_column) for the top p projection columns.
    - filtering_columns_with_tables: List of tuples (table_name, filtering_column) for the top w filtering columns.
    """
    # Step 1: Retrieve avg_projections_per_query (p) and avg_conditions_per_query (w)
    avg_projection = df.loc[df['db_id'] == db_id, 'avg_projections_per_query'].values[0]
    avg_conditions = df.loc[df['db_id'] == db_id, 'avg_conditions_per_query'].values[0]
    
    p = int(avg_projection)  # Number of projection columns to include
    w = int(avg_conditions)  # Number of filtering columns to include
    print(f"Using p={p} and w={w} for the database '{db_id}'")
    
    # Step 2: Filter rows where table_name is in tables_involved
    tables_involved = [table.lower() for table in tables_involved]
    involved_tables_df = df[(df['db_id'] == db_id) & (df['table_name'].isin(tables_involved))].copy()
    
    # Step 3: Handle NaN values and split occurrence strings
    involved_tables_df['projection_occurrences'] = involved_tables_df['projection_occurrences'].fillna('0/1')
    involved_tables_df['projection_occurrences'] = involved_tables_df['projection_occurrences'].str.split('/').str[0].astype(int)
    
    # Step 4: Select the top p projection columns based on projection_occurrences
    top_projection_columns_df = involved_tables_df.sort_values(by='projection_occurrences', ascending=False).head(p)
    # Get the projection columns with their corresponding tables
    projection_columns_with_tables = list(top_projection_columns_df[['table_name', 'projection_column']].itertuples(index=False, name=None))
    
    # Step 5: Handle filtering columns while excluding those in the projection columns
    if 'filtering_column' in involved_tables_df.columns:
        involved_tables_df['filtering_occurrences'] = involved_tables_df['filtering_occurrences'].fillna('0/1')
        involved_tables_df['filtering_occurrences'] = involved_tables_df['filtering_occurrences'].str.split('/').str[0].astype(int)
        
        # Exclude columns that are in the projection from filtering columns
        projection_columns = set(col[1] for col in projection_columns_with_tables)  # Get the names of projection columns
        filtering_candidates_df = involved_tables_df[~involved_tables_df['filtering_column'].isin(projection_columns)]
        
        # Select the top w filtering columns based on filtering_occurrences
        top_filtering_columns_df = filtering_candidates_df.sort_values(by='filtering_occurrences', ascending=False).head(w)
        filtering_columns_with_tables = list(top_filtering_columns_df[['table_name', 'filtering_column']].itertuples(index=False, name=None))
    else:
        filtering_columns_with_tables = []

    return projection_columns_with_tables, filtering_columns_with_tables


In [138]:
def generate_inner_join_query_with_test(schema, pattern, db_id, df):
    """
    Generate an INNER JOIN SQL query and a test query for validation.

    Parameters:
    - schema: The graph schema containing table information.
    - pattern: The subgraph pattern to generate the query for.
    - original_nodes: Nodes in the original subgraph.
    - new_nodes: Newly added nodes in the extended subgraph.
    - db_id: The database ID to filter on for projection/filtering columns.
    - df: The DataFrame containing information about projection and filtering columns.

    Returns:
    - A dictionary with the original query, the test query, projection columns, filtering columns.
    """
    tables = set()
    join_conditions = []

    # Collect table data for each node in the subgraph using the schema
    for node in pattern.nodes():
        table_name, columns, _ = get_table_data(schema, node)
        if table_name:
            tables.add(table_name)

    # Collect join conditions from the edges of the subgraph
    for edge in pattern.edges(data=True):
        left_table, right_table, edge_data = edge
        if 'label' in edge_data:
            join_condition = edge_data.get('label')
        else:
            join_condition = schema.edges[(left_table, right_table)].get('label')
        join_conditions.append((left_table, right_table, join_condition))

    # Get the tables involved in the current query
    tables_involved = list(tables)

    # Retrieve projection and filtering columns based on the db_id and involved tables
    projection_columns_with_tables, filtering_columns_with_tables = retrieve_projection_filter_columns(db_id, df, tables_involved)
    # Construct the SELECT columns for the main query
    select_columns = [exp.Column(this=col, table=table) for table, col in projection_columns_with_tables]

    # Initialize the base query
    base_query = sqlglot.select(*select_columns).distinct()  # Add DISTINCT to the main query

    # Construct the FROM clause with all involved tables
    from_clause = ', '.join(tables_involved)
    base_query = base_query.from_(from_clause)

    # Prepare WHERE conditions, starting with join conditions
    where_conditions = []

    # Add join conditions to WHERE
    for _, _, condition in join_conditions:
        if len(condition.split(';')) > 1:
            condition = condition.split(';')[0]
        left_col = condition.split('=')[0].strip().split('.')[1]
        left_table = condition.split('=')[0].strip().split('.')[0]
        right_col = condition.split('=')[1].strip().split('.')[1]
        right_table = condition.split('=')[1].strip().split('.')[0]
        where_conditions.append(exp.EQ(
            this=exp.Column(this=left_col, table=left_table),
            expression=exp.Column(this=right_col, table=right_table)
        ))

    # Add filtering conditions to WHERE (using placeholders)
    for table, col in filtering_columns_with_tables:
        where_conditions.append(exp.EQ(
            this=exp.Column(this=col, table=table),
            expression=exp.Placeholder()  # Placeholder for filtering
        ))

    # Apply WHERE conditions without unnecessary parentheses
    if where_conditions:
        base_query = base_query.where(where_conditions[0])  # Start with the first condition
        for condition in where_conditions[1:]:
            base_query = base_query.where(condition)  # Add remaining conditions

    # Replace parentheses with an empty character in the SQL query string
    main_query = base_query.sql().replace('(', '').replace(')', '')

    # Now, generate the test query
    test_select_columns = [f"{table}.{col}" for table, col in projection_columns_with_tables + filtering_columns_with_tables]
    
    # Create the WHERE clause for the test query
    test_where_conditions = []
    for _, _, condition in join_conditions:
        if len(condition.split(';')) > 1:
            condition = condition.split(';')[0]  # Use only the first part if there are multiple
        left_table, left_col = condition.split('=')[0].strip().split('.')
        right_table, right_col = condition.split('=')[1].strip().split('.')
        test_where_conditions.append(f"{left_table}.{left_col} = {right_table}.{right_col}")


    # Construct the test query
    test_query = f"""
    SELECT {', '.join(test_select_columns)}, COUNT(*) AS combination_count
    FROM {from_clause}
    WHERE {' AND '.join(test_where_conditions)}
    GROUP BY {', '.join(test_select_columns)};
    """

    # Return both queries and relevant columns
    return {
        'main_query': main_query,
        'test_query': test_query,
        'projection_columns': projection_columns_with_tables,
        'filtering_columns': filtering_columns_with_tables
    }



In [139]:
def process_rule_data(rule_patterns, schema, db_id, df, rule_name, financial_table_fix=False):
    """
    Process rule data to generate SQL queries for subgraphs.

    Parameters:
    - rule_patterns: List of patterns (subgraphs) for the rule.
    - schema: The graph schema containing table information.
    - db_id: The database ID for filtering projection/filtering columns.
    - df: The DataFrame with information on projection and filtering columns.
    - rule_name: The name of the rule (e.g., 'rule_1', 'rule_2').
    - financial_table_fix: If True, apply fixes for reserved keywords in SQL queries.

    Returns:
    - List of dictionaries containing processed subgraph data.
    """
    rule_data = []
    generated_queries = []

    # Check if the rule_patterns list is empty
    if not rule_patterns:
        # If empty, create a dict with None values and return immediately
        rule_data.append({
            'subgraph': None,
            'main_query': None,
            'test_query': None,
            'projection_columns': None,
            'filtering_columns': None
        })
        return rule_data

    for i, subgraph in enumerate(rule_patterns, start=1):
        # Generate queries for the subgraph
        result = generate_inner_join_query_with_test(schema, subgraph, db_id, df)

        # Apply fixes for reserved keywords in SQL queries (if required)
        if financial_table_fix and db_id == 'financial':
            if 'order' in result['main_query']:
                result['main_query'] = result['main_query'].replace('order', "'order'")
                result['test_query'] = result['test_query'].replace('order', "'order'")

        # Store the results in a dictionary
        rule_datum = {
            'subgraph': subgraph,
            'main_query': result['main_query'],
            'test_query': result['test_query'],
            'projection_columns': result['projection_columns'],
            'filtering_columns': result['filtering_columns']
        }
        rule_data.append(rule_datum)
        generated_queries.append(result)

        # Print the queries for debugging or logging
        print(f"Subgraph {i} Query:\nMain Query:\n{result['main_query']}\nTest Query:\n{result['test_query']}\n")

    return rule_data


# Process Rule 1 Data
rule_1_data = process_rule_data(
    rule_patterns=rule_1_sorted_patterns,
    schema=schema,
    db_id=db_id,
    df=df,
    rule_name='rule_1',
    financial_table_fix=True
)

# Process Rule 2 Data
rule_2_data = process_rule_data(
    rule_patterns=rule_2_sorted_patterns,
    schema=schema,
    db_id=db_id,
    df=df,
    rule_name='rule_2',
    financial_table_fix=True
)


Using p=1 and w=1 for the database 'toxicology'
Subgraph 1 Query:
Main Query:
SELECT DISTINCT molecule.molecule_id FROM atom, bond, molecule, connected WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id AND bond.bond_type = ?
Test Query:

    SELECT molecule.molecule_id, bond.bond_type, COUNT(*) AS combination_count
    FROM atom, bond, molecule, connected
    WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id
    GROUP BY molecule.molecule_id, bond.bond_type;
    

Using p=1 and w=1 for the database 'toxicology'
Subgraph 2 Query:
Main Query:
SELECT DISTINCT molecule.molecule_id FROM bond, atom, molecule, connected WHERE connected.atom_id2 = atom.atom_id AND connected.bond_id = bond.bond_id AND atom.molecule_id = molecule.molecule_id AND bond.bond_type = ?
Test Query:

    SELECT molecule.molecule_id, bond.bond_type, COUNT(*) AS combination_cou

In [140]:
print(rule_2_data)

[{'subgraph': <networkx.classes.graph.Graph object at 0x312b34dd0>, 'main_query': 'SELECT DISTINCT molecule.molecule_id FROM bond, molecule, atom WHERE molecule.molecule_id = atom.molecule_id AND bond.molecule_id = molecule.molecule_id AND atom.molecule_id = bond.molecule_id AND atom.element = ?', 'test_query': '\n    SELECT molecule.molecule_id, atom.element, COUNT(*) AS combination_count\n    FROM bond, molecule, atom\n    WHERE molecule.molecule_id = atom.molecule_id AND bond.molecule_id = molecule.molecule_id AND atom.molecule_id = bond.molecule_id\n    GROUP BY molecule.molecule_id, atom.element;\n    ', 'projection_columns': [('molecule', 'molecule_id')], 'filtering_columns': [('atom', 'element')]}]


## 5. Replacing placeholders by values from database files
The database files used in this cell are provided by the BIRD benchmark for the development databases. They can be found here: [BIRD Development Databases](https://drive.google.com/drive/folders/1Omlu6On1trI9j2c8V2Ip4w02cEYyiNbs).


In [141]:
import sqlite3

def vacuum_database(conn):
    """Vacuum the SQLite database to optimize performance."""
    cursor = conn.cursor()
    try:
        cursor.execute("VACUUM;")
        print("Database vacuumed successfully.")
    except Exception as e:
        print(f"Error during vacuuming: {e}")
    finally:
        cursor.close()  # Close the cursor but keep the connection open

def execute_test_query_and_replace_placeholders(conn, test_query, main_query, filtering_columns):
    """
    Execute a test query and replace placeholders in the main query with actual filtering values.
    """
    cursor = conn.cursor()
    try:
        # Execute the test query
        print("Executing test query...")
        print("test_query:", test_query)
        cursor.execute(test_query)

        # Fetch all results to determine the maximum COUNT(*) and corresponding filtering values
        results = cursor.fetchall()
        max_count = 0
        max_values = None

        for row in results:
            count_value = row[-1]  # The last column is COUNT(*)
            if count_value > max_count:
                max_count = count_value
                max_values = row[-(len(filtering_columns) + 1):-1]  # All columns except the last one

        # If no result is found, return the main query unchanged
        if max_values is None:
            print(f"No results found for test query. Returning original main query.")
            return main_query

        # Replace placeholders in the main query with actual values
        for value in max_values:
            main_query = main_query.replace('?', f"'{value}'", 1)  # Replace one placeholder at a time

    finally:
        cursor.close()  # Ensure the cursor is closed after use

    return main_query

def process_queries_for_rules(rule_data, db_id, database_path):
    """
    Process rule data by executing test queries and replacing placeholders in main queries.

    Parameters:
    - rule_data: List of dictionaries containing subgraph and query information.
    - db_id: Database ID to construct the database path.
    - database_path: Path template for accessing SQLite databases.
    """
    for i, result in enumerate(rule_data, start=1):
        print(f"Processing subgraph {i}...")

        # Check for None values in necessary fields before processing
        test_query = result.get('test_query')
        main_query = result.get('main_query')
        filtering_columns = result.get('filtering_columns')

        # If any of the required fields are None, skip the processing for this result
        if test_query is None or main_query is None or filtering_columns is None:
            print(f"Skipping this list due to missing data (None values).")
            return result

        # Open a new connection for each query
        with sqlite3.connect(database_path.format(db_id=db_id)) as conn:
            # Vacuum the database before processing queries
            vacuum_database(conn)

            # Execute the test query and replace placeholders in the main query
            final_query = execute_test_query_and_replace_placeholders(conn, test_query, main_query, filtering_columns)

        # Update the rule data with the final query (in-place update)
        result['main_query'] = final_query

        print(f"Updated main_query for Subgraph {i}:\n{final_query}\n")

# Example usage for both rule_1_data and rule_2_data
database_path = 'dev_databases/{db_id}/{db_id}.sqlite'

# Process Rule 1 Data and update it directly
process_queries_for_rules(rule_1_data, db_id, database_path)

# Process Rule 2 Data and update it directly
process_queries_for_rules(rule_2_data, db_id, database_path)


Processing subgraph 1...
Database vacuumed successfully.
Executing test query...
test_query: 
    SELECT molecule.molecule_id, bond.bond_type, COUNT(*) AS combination_count
    FROM atom, bond, molecule, connected
    WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id
    GROUP BY molecule.molecule_id, bond.bond_type;
    
Updated main_query for Subgraph 1:
SELECT DISTINCT molecule.molecule_id FROM atom, bond, molecule, connected WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id AND bond.bond_type = '-'

Processing subgraph 2...
Database vacuumed successfully.
Executing test query...
test_query: 
    SELECT molecule.molecule_id, bond.bond_type, COUNT(*) AS combination_count
    FROM bond, atom, molecule, connected
    WHERE connected.atom_id2 = atom.atom_id AND connected.bond_id = bond.bond_id AND atom.molecule_id = molecule.molecule_id
    GR

## 6. Validating the generated queries

In [142]:
import sqlite3
import pandas as pd

def execute_query(db_file, query):
    """
    Connects to the SQLite database and executes the given query.

    Parameters:
    - db_file: Path to the SQLite database file.
    - query: SQL query to execute.

    Returns:
    - DataFrame containing the results of the query.
    """
    # Connect to the database
    conn = sqlite3.connect(db_file)
    
    try:
        # Execute the query and fetch results
        df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()  # Return an empty DataFrame on error
    finally:
        # Close the connection
        conn.close()

def filter_valid_queries(rule_data, db_file):
    """
    Filters the given rule data to keep only valid queries.

    Parameters:
    - rule_data: List of dictionaries containing query information.
    - db_file: Path to the SQLite database file.

    Returns:
    - List of dictionaries with valid queries.
    """
    valid_data = []

    if any(element['main_query'] == None for element in rule_data):
        print(f"Total Queries: {0}")
    else:
        print(f"Total Queries: {len(rule_data)}")
    for element in rule_data:
        if element['main_query'] == None:
            return rule_data
        
        else:
            # Execute the query and check if it returns results
            result_df = execute_query(db_file, element['main_query'])
            if not result_df.empty:  # Check if the query returned results
                print("Query executed successfully!")
                valid_data.append(element)
    return valid_data

# Example usage:

db_file = f'dev_databases/{db_id}/{db_id}.sqlite'  # Replace with your actual database file


# Filter valid queries for both rules
rule_1_data = filter_valid_queries(rule_1_data, db_file)
rule_2_data = filter_valid_queries(rule_2_data, db_file)

# Optionally, print the valid queries
print("Valid Rule 1 Data:")
for valid_query in rule_1_data:
    print(valid_query)

print("Valid Rule 2 Data:")
for valid_query in rule_2_data:
    print(valid_query)



Total Queries: 2
Query executed successfully!
Query executed successfully!
Total Queries: 1
Query executed successfully!
Valid Rule 1 Data:
{'subgraph': <networkx.classes.graph.Graph object at 0x313a3ba10>, 'main_query': "SELECT DISTINCT molecule.molecule_id FROM atom, bond, molecule, connected WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id AND bond.bond_type = '-'", 'test_query': '\n    SELECT molecule.molecule_id, bond.bond_type, COUNT(*) AS combination_count\n    FROM atom, bond, molecule, connected\n    WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id\n    GROUP BY molecule.molecule_id, bond.bond_type;\n    ', 'projection_columns': [('molecule', 'molecule_id')], 'filtering_columns': [('bond', 'bond_type')]}
{'subgraph': <networkx.classes.graph.Graph object at 0x16fc055d0>, 'main_query': "SELECT DISTINCT molecule.molecule_id FROM bond,

## 6. Saving Rule 1 and 2 outputs

In [143]:
import os
import pickle
import pandas as pd

# Path setup
base_folder = 'rules_outputs'
rule_1_folder = os.path.join(base_folder, 'rule_1')
rule_2_folder = os.path.join(base_folder, 'rule_2')

# Read the CSV for number of queries
df = pd.read_csv('query_statistics.csv')
number_of_queries = df.loc[df['db_id'] == db_id, 'num_queries'].values[0]

# Create directories if they don't exist
os.makedirs(rule_1_folder, exist_ok=True)
os.makedirs(rule_2_folder, exist_ok=True)

# Function to process and save the entire list of data
def save_rule_data(rule_data, folder, db_id, number_of_queries):
    # Vérifier si un seul dictionnaire avec des valeurs None existe
    if any(d['subgraph'] is None and d['main_query'] is None and d['test_query'] is None and
           d['projection_columns'] is None and d['filtering_columns'] is None for d in rule_data):
        # Ajouter les champs par défaut si un dict avec None est trouvé
        for data_dict in rule_data:
            if data_dict['subgraph'] is None and data_dict['main_query'] is None and data_dict['test_query'] is None and \
               data_dict['projection_columns'] is None and data_dict['filtering_columns'] is None:
                data_dict['number_of_added_queries'] = 0
                data_dict['number_of_queries'] = 0
                data_dict['ratio_of_added_queries'] = 0.0
    else:
        # Sinon, ajouter les champs calculés pour tous les dicts dans la liste
        k_value = len(rule_data)
        for data_dict in rule_data:
            data_dict['number_of_added_queries'] = k_value
            data_dict['number_of_queries'] = number_of_queries
            # Calculate the ratio of suggested queries to the total number of queries
            data_dict['ratio_of_added_queries'] = (k_value / number_of_queries) * 100 if number_of_queries > 0 else 0.0

    # Create output file path
    output_file = os.path.join(folder, f"{db_id}_outputs.pkl")
    
    # Save the entire list to a pickle file
    with open(output_file, 'wb') as f:
        pickle.dump(rule_data, f)


save_rule_data(rule_1_data, rule_1_folder, db_id, number_of_queries)
save_rule_data(rule_2_data, rule_2_folder, db_id, number_of_queries)

print("After saving:")
print("Rule 1 Data:", rule_1_data)
print("Rule 2 Data:", rule_2_data)
print("Data saved successfully.")


After saving:
Rule 1 Data: [{'subgraph': <networkx.classes.graph.Graph object at 0x313a3ba10>, 'main_query': "SELECT DISTINCT molecule.molecule_id FROM atom, bond, molecule, connected WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id AND bond.bond_type = '-'", 'test_query': '\n    SELECT molecule.molecule_id, bond.bond_type, COUNT(*) AS combination_count\n    FROM atom, bond, molecule, connected\n    WHERE connected.bond_id = bond.bond_id AND atom.molecule_id = bond.molecule_id AND bond.molecule_id = molecule.molecule_id\n    GROUP BY molecule.molecule_id, bond.bond_type;\n    ', 'projection_columns': [('molecule', 'molecule_id')], 'filtering_columns': [('bond', 'bond_type')], 'number_of_added_queries': 2, 'number_of_queries': 128, 'ratio_of_added_queries': 1.5625}, {'subgraph': <networkx.classes.graph.Graph object at 0x16fc055d0>, 'main_query': "SELECT DISTINCT molecule.molecule_id FROM bond, atom, molecule, conn

## 7. Global ordering of all generated queries
This cell should only be run after executing the previous cells for each `db_id` in the development set.

In [144]:
import os
import pickle
import json
import networkx as nx

# Function to calculate subgraph centrality score
def calculate_subgraph_centrality(subgraph, centrality_dict):
    subgraph_nodes = list(subgraph.nodes())
    return sum(centrality_dict.get(node, 0) for node in subgraph_nodes) / len(subgraph_nodes)

# Function to check if a subgraph is cyclic
def is_cyclic(subgraph):
    try:
        nx.find_cycle(subgraph)
        return True
    except nx.NetworkXNoCycle:
        return False

# Function to process and compute metrics for all databases in a rule folder
def process_rule_folder(rule_folder, output_json_file):
    # Collect all pickle files in the folder
    files = [f for f in os.listdir(rule_folder) if f.endswith('_outputs.pkl')]
    
    global_rule_data = []

    # Process each pickle file
    for file in files:
        # Extract db_id by removing '_outputs.pkl'
        db_id = file.replace('_outputs.pkl', '')  
        schema_path = f'dev_dbs_unnested/main_graphs_pickles/{db_id}_graph.pkl'

        # Load the schema graph
        with open(schema_path, 'rb') as f:
            schema = pickle.load(f)
        
        # Compute centrality for each node in the schema
        centrality = nx.degree_centrality(schema)

        # Load the rule data from pickle file
        with open(os.path.join(rule_folder, file), 'rb') as f:
            data_list = pickle.load(f)

        # Compute metrics for each subgraph in the pickle data
        for data in data_list:
            subgraph = data.get('subgraph', None)

            # If there's no subgraph, skip this entry
            if subgraph is None:
                continue

            # Extract and calculate subgraph metrics
            num_nodes = subgraph.number_of_nodes()
            num_connections = subgraph.number_of_edges()
            cyclic = is_cyclic(subgraph)
            centrality_score = calculate_subgraph_centrality(subgraph, centrality)
            centrality_score = round(float(centrality_score), 3)

            # Ensure the ratio is rounded and serializable
            ratio_of_added_queries = round(float(data['ratio_of_added_queries']), 3)

            # Append relevant data to the global list, with metrics
            global_rule_data.append({
                'db_id': db_id,
                'main_query': data['main_query'],
                'number_of_added_queries': int(data['number_of_added_queries']),  # Ensure k_value is an int
                'ratio_of_added_queries': ratio_of_added_queries,  # Rounded ratio
                'number_of_queries': int(data['number_of_queries']),  # Ensure it's an int
                'question': "",  # Placeholder
                'num_nodes': num_nodes,
                'num_connections': num_connections,
                'cyclic': cyclic,
                'centrality_score': centrality_score
            })

    # Sort all data globally in descending order based on the subgraph metrics
    global_rule_data_sorted = sorted(
        global_rule_data, 
        key=lambda x: (x['num_nodes'], x['num_connections'], x['cyclic'], x['centrality_score']),
        reverse=True  # Sort from biggest to smallest
    )

    # Create output JSON
    with open(output_json_file, 'w') as json_file:
        json.dump(global_rule_data_sorted, json_file, indent=4)

# Define paths
base_folder = 'rules_outputs'
rule_1_folder = os.path.join(base_folder, 'rule_1')
rule_2_folder = os.path.join(base_folder, 'rule_2')

# Process rule 1 and rule 2 data and generate corresponding JSON files
process_rule_folder(rule_1_folder, 'rule_1_ordered.json')
process_rule_folder(rule_2_folder, 'rule_2_ordered.json')

print("Global ordering for all databases done and JSON files created successfully.")

Global ordering for all databases done and JSON files created successfully.


## 8. Final statistics 
This cell provides a summary of statistics for the outputs of Rule 1 and Rule 2. This summary can be ignored for now.


In [145]:
import os
import pickle
import csv
import pandas as pd

# Function to process the pickle files for a rule and store db_id, k_value, and rule in a CSV
def process_rule_to_csv(rule_folder, rule_name, csv_writer, unique_rows):
    # Collect all pickle files in the folder
    files = [f for f in os.listdir(rule_folder) if f.endswith('_outputs.pkl')]

    # Process each pickle file
    for file in files:
        # Extract db_id by removing '_outputs.pkl'
        db_id = file.replace('_outputs.pkl', '')

        # Load the rule data from pickle file
        with open(os.path.join(rule_folder, file), 'rb') as f:
            data_list = pickle.load(f)

        # Write db_id, k_value, and rule for each dictionary in the pickle data
        for data in data_list:
            db_id_value = db_id
            k_value = data.get('number_of_added_queries', 0)  # Default k_value to 0 if not present

            df = pd.read_csv('query_statistics.csv')
            number_of_queries = float(df.loc[df['db_id'] == db_id, 'num_queries'].values[0])
            ratio_of_added_queries = round(((k_value / number_of_queries) if number_of_queries > 0 else 0.0) * 100, 3)
            # Create a tuple of the row
            row = (db_id_value, k_value, rule_name, number_of_queries, ratio_of_added_queries)

            # Only write the row if it's unique
            if row not in unique_rows:
                unique_rows.add(row)
                csv_writer.writerow(row)

# Define paths
base_folder = 'rules_outputs'
rule_1_folder = os.path.join(base_folder, 'rule_1')
rule_2_folder = os.path.join(base_folder, 'rule_2')

# CSV file path
csv_file_path = 'rule_data_summary.csv'

# Open CSV file and write headers
with open(csv_file_path, mode='w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)

    # Write the header row
    csv_writer.writerow(['db_id', 'number_of_added_queries', 'rule', 'number_of_unnested_queries', 'ratio_of_added_queries'])

    # Set to track unique rows
    unique_rows = set()

    # Process rule 1 data and write to CSV
    process_rule_to_csv(rule_1_folder, 'rule_1', csv_writer, unique_rows)

    # Process rule 2 data and write to CSV
    process_rule_to_csv(rule_2_folder, 'rule_2', csv_writer, unique_rows)

print(f"Data successfully written to {csv_file_path}.")


Data successfully written to rule_data_summary.csv.
